# 00 · Setup and Data Ingestion to S3

**Steam Store Copilot** keeps all raw data in **Amazon S3**, not in GitHub. This notebook moves the five raw files into
`s3://<sagemaker-default-bucket>/steam-store-copilot/raw/`.

* If the `steam-data/` folder was uploaded to SageMaker Studio, the files are copied from there.
* Any file not found locally is downloaded from Kaggle (needs `~/.kaggle/kaggle.json`).

| File | Source / License | Rows |
|---|---|---|
| `store/games.csv` | Fronkon Games — Steam Games Dataset (MIT) | 125,855 |
| `recs/games.csv` | Game Recommendations on Steam (CC0) | 50,872 |
| `recs/users.csv` | same | ~14M |
| `recs/recommendations.csv` | same | 41,154,794 |
| `recs/games_metadata.json` | same | 50,872 |

In [1]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.path.expanduser("~/steam-store-copilot")]
    try:  # the kernel's working directory may have been deleted (e.g. after re-cloning)
        candidates = [os.getcwd(), os.path.dirname(os.getcwd())] + candidates
    except FileNotFoundError:
        os.chdir(os.path.expanduser("~"))
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import logging

for _name in ("sagemaker", "sagemaker.config", "botocore.credentials"):
    logging.getLogger(_name).setLevel(logging.ERROR)  # keep SDK log noise out of the outputs

import boto3
import sagemaker

from src.config import S3Layout

# Works both inside SageMaker Studio and on a laptop with Learner Lab credentials in ~/.aws/credentials
boto_session = boto3.Session(region_name=boto3.Session().region_name or "us-east-1")
sess = sagemaker.Session(boto_session=boto_session)
region = sess.boto_region_name
if os.path.exists("/opt/ml/metadata/resource-metadata.json"):  # inside SageMaker Studio
    role = sagemaker.get_execution_role(sagemaker_session=sess)
else:  # laptop with Learner Lab credentials: jobs run as the lab's execution role
    account = boto_session.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/LabRole"
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

Repo root: /Users/yatharthvardan/Documents/USD/steam-store-copilot


Region: us-east-1
Role:   arn:aws:iam::326278567375:role/LabRole
Bucket: sagemaker-us-east-1-326278567375


In [3]:
import pandas as pd

from src.data.s3io import RAW_FILES, list_objects

CANDIDATE_DIRS = [
    os.path.expanduser("~/steam-data"),
    os.path.join(REPO_ROOT, "steam-data"),
    os.path.join(REPO_ROOT, "..", "steam-data"),
]


def find_local(rel):
    for root in CANDIDATE_DIRS:
        path = os.path.abspath(os.path.join(root, rel))
        if os.path.exists(path):
            return path
    return None


local = {rel: find_local(rel) for rel in RAW_FILES}
pd.DataFrame([{"file": rel, "local_path": p, "size_mb": round(os.path.getsize(p) / 1e6, 1) if p else None,
               "description": RAW_FILES[rel]} for rel, p in local.items()])

,file,local_path,size_mb,description
0,store/games.csv,/Users/yatharthvardan/Documents/USD/steam-data...,401.1,Fronkon Games Steam Games Dataset (MIT) - desc...
1,recs/games.csv,/Users/yatharthvardan/Documents/USD/steam-data...,4.9,Game Recommendations on Steam (CC0) - game rat...
2,recs/users.csv,/Users/yatharthvardan/Documents/USD/steam-data...,193.1,Game Recommendations on Steam (CC0) - reviewer...
3,recs/recommendations.csv,/Users/yatharthvardan/Documents/USD/steam-data...,2023.2,Game Recommendations on Steam (CC0) - 41M user...
4,recs/games_metadata.json,/Users/yatharthvardan/Documents/USD/steam-data...,17.5,Game Recommendations on Steam (CC0) - descript...


## Upload local files to S3
Files already in S3 with the same size are skipped, so this cell is safe to re-run.

In [4]:
from boto3.s3.transfer import TransferConfig

s3 = boto3.client("s3")
transfer = TransferConfig(multipart_chunksize=64 * 1024 * 1024, max_concurrency=8)


def s3_size(key):
    try:
        return s3.head_object(Bucket=layout.bucket, Key=key)["ContentLength"]
    except s3.exceptions.ClientError:
        return None


for rel, path in local.items():
    key = f"{layout.prefix}/raw/{rel}"
    if path is None:
        continue
    if s3_size(key) == os.path.getsize(path):
        print(f"skip   {rel} (already in S3)")
        continue
    print(f"upload {rel} ...", end=" ", flush=True)
    s3.upload_file(path, layout.bucket, key, Config=transfer)
    print("done")

skip   store/games.csv (already in S3)
skip   recs/games.csv (already in S3)
skip   recs/users.csv (already in S3)


skip   recs/recommendations.csv (already in S3)
skip   recs/games_metadata.json (already in S3)


## Kaggle fallback for anything still missing
Upload your `kaggle.json` to `~/.kaggle/kaggle.json` first. Each file is downloaded, uploaded to S3, and deleted locally
so the Studio volume never holds more than one file at a time.

In [5]:
import subprocess
import tempfile
import zipfile

KAGGLE = {
    "store/games.csv": ("fronkongames/steam-games-dataset", "games.csv"),
    "recs/games.csv": ("antonkozyriev/game-recommendations-on-steam", "games.csv"),
    "recs/users.csv": ("antonkozyriev/game-recommendations-on-steam", "users.csv"),
    "recs/recommendations.csv": ("antonkozyriev/game-recommendations-on-steam", "recommendations.csv"),
    "recs/games_metadata.json": ("antonkozyriev/game-recommendations-on-steam", "games_metadata.json"),
}

missing = [rel for rel in RAW_FILES if s3_size(f"{layout.prefix}/raw/{rel}") is None]
print("Missing in S3:", missing or "none")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    for rel in missing:
        dataset, fname = KAGGLE[rel]
        with tempfile.TemporaryDirectory() as tmp:
            subprocess.run(["kaggle", "datasets", "download", dataset, "-f", fname, "-p", tmp], check=True)
            path = os.path.join(tmp, fname)
            if not os.path.exists(path):
                with zipfile.ZipFile(path + ".zip") as zf:
                    zf.extractall(tmp)
                os.remove(path + ".zip")
            s3.upload_file(path, layout.bucket, f"{layout.prefix}/raw/{rel}", Config=transfer)
            print("uploaded", rel)

Missing in S3: none


## Raw data inventory in S3

In [6]:
inventory = list_objects(layout.raw)
inventory

,key,size_mb,last_modified
0,steam-store-copilot/raw/recs/games.csv,4.9,2026-10-17 15:28:43+00:00
1,steam-store-copilot/raw/recs/games_metadata.json,17.5,2026-10-17 15:29:15+00:00
2,steam-store-copilot/raw/recs/recommendations.csv,2023.2,2026-10-17 15:29:06+00:00
3,steam-store-copilot/raw/recs/users.csv,193.1,2026-10-17 15:28:46+00:00
4,steam-store-copilot/raw/store/games.csv,401.1,2026-10-17 15:28:41+00:00


In [7]:
assert len(inventory) == len(RAW_FILES), "Some raw files are missing from S3"
print("All raw files are in", layout.raw)

All raw files are in s3://sagemaker-us-east-1-326278567375/steam-store-copilot/raw
